# 🚀 Gemma SFT Fine-Tuning v2 (`fine-tuning-agent-on-traces-v2`)

This production-grade v2 notebook performs end-to-end Supervised Fine-Tuning of **Gemma 2B-it** on agent execution traces from `badlogicgames/pi-mono` using **Unsloth 4-bit QLoRA** and **Completion-Only Loss**:

### 🛠️ Systematic Root-Cause Fixes in v2:
1. **Unsloth Official 4-bit Pre-Quantized Model**: Uses `MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"` (whose native config has `torch_dtype: float16`), and sets `fp16=False, bf16=False` in `TrainingArguments` to allow Unsloth to manage internal kernel precision natively without PyTorch `autocast` dtype conflicts (`RuntimeError: self and mat2 must have the same dtype`).
2. **GPU Direct Dispatch**: Uses `device_map="cuda"` to force 4-bit model quantization directly onto GPU VRAM, preventing CPU offloading `ValueError`.
3. **Processor/Tokenizer Extraction**: Safely extracts the text tokenizer from `Processor` objects to eliminate `AttributeError: 'Gemma4Processor' object has no attribute 'encode'`.
4. **Robust Completion-Only Collator**: Implements a PyTorch-native `DataCollatorForCompletionOnlyLM` that accepts tokenized feature dictionaries (`input_ids`, `attention_mask`), pads with `tokenizer.pad()`, and masks prompt tokens (`labels[:response_start] = -100`).
5. **Schema-Agnostic JSONL Parser**: Replaces PyArrow schema inference with native Python `json.loads` to eliminate `DatasetGenerationCastError` across heterogeneous trace files.
6. **Trace & Tool Call Extraction**: Strips internal thinking/reasoning parts and structures trace message turns.
7. **API & Callback Compatibility**: Uses `eval_strategy="steps"`, `report_to="none"`, and extracts `held_out_eval_loss` directly from `trainer.state.log_history` to prevent `RuntimeError` callbacks.
8. **VRAM Garbage Collection**: Explicitly cleans PyTorch CUDA memory (`torch.cuda.empty_cache()`) between sweep jobs and before evaluation to prevent OOM errors.
9. **Repository Target**: Publishes adapters and final merged model weights to `<HF_USERNAME>/fine-tuning-agent-on-traces-v2`.

## Cell 1: Install Dependencies & Set Hugging Face Token

In [ ]:
# Install Unsloth, TRL, PEFT, TrackIO, Inspect AI, and Hugging Face Hub
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes trackio inspect-ai datasets huggingface_hub

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

# Set Hugging Face Token (Add HF_TOKEN in Colab Secrets or paste token string below)
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = "YOUR_HF_TOKEN_HERE"  # <-- Replace with your HF Write Token if not in Colab Secrets

os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

# Verify Hugging Face Authentication & Target Repo v2
api = HfApi()
user_info = api.whoami()
HF_USERNAME = user_info['name']
FINAL_REPO_NAME = f"{HF_USERNAME}/fine-tuning-agent-on-traces-v2"
TRACKIO_PROJECT = "youtube-livestream-1"

print(f"✅ Authenticated as HF User: {HF_USERNAME}")
print(f"Final Model Target Repo: https://huggingface.co/{FINAL_REPO_NAME}")

## Cell 2: Preprocess `badlogicgames/pi-mono` Dataset for Gemma

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
from datasets import Dataset
import json

print("Downloading and parsing jsonl trace files from badlogicgames/pi-mono...")
api = HfApi()
repo_files = api.list_repo_files('badlogicgames/pi-mono', repo_type='dataset')
jsonl_files = [f for f in repo_files if f.endswith('.jsonl')]

SYSTEM_PROMPT = "You are an expert AI software engineering agent trained on pi-mono execution traces. Your task is to analyze codebase context, follow step-by-step reasoning, execute agentic workflow actions, and generate accurate code implementations."

raw_messages = []
# Download and extract trace messages directly without PyArrow casting errors
for f in jsonl_files[:60]:  # Parse sample of trace files
    try:
        fpath = hf_hub_download(repo_id='badlogicgames/pi-mono', filename=f, repo_type='dataset')
        with open(fpath, 'r', encoding='utf-8') as fp:
            for line in fp:
                if line.strip():
                    item = json.loads(line)
                    msg = item.get('message', {})
                    role = msg.get('role', 'user')
                    content = msg.get('content', '')
                    if isinstance(content, list):
                        content_str = '\n'.join([
                            c.get('text', '') if isinstance(c, dict) else str(c)
                            for c in content
                            if isinstance(c, dict) and c.get('type') != 'thinking'
                        ])
                    else:
                        content_str = str(content)
                    if content_str.strip():
                        raw_messages.append({'role': role, 'content': content_str.strip()})
    except Exception:
        pass

# Pre-format into Gemma chat prompt strings
formatted_prompts = []
for i in range(0, len(raw_messages) - 1, 2):
    user_text = raw_messages[i]['content'][:2048]
    assistant_text = raw_messages[i+1]['content'][:2048]
    if user_text and assistant_text:
        text = f"<start_of_turn>user\nSystem: {SYSTEM_PROMPT}\n\nTask: {user_text}<end_of_turn>\n<start_of_turn>model\n{assistant_text}<end_of_turn>"
        formatted_prompts.append({'text': text})

dataset = Dataset.from_list(formatted_prompts)
split_ds = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split_ds["train"]
eval_ds = split_ds["test"]

print(f"✅ Successfully formatted dataset for Gemma SFT!")
print(f"Train records: {len(train_ds)}, Evaluation records: {len(eval_ds)}")

## Cell 3: Parameter Sweep Setup & Training Loop (Unsloth + Completion-Only Loss + TrackIO)

In [ ]:
import trackio
import torch
import gc
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# PyTorch-native DataCollatorForCompletionOnlyLM supporting tokenized batch features
class DataCollatorForCompletionOnlyLM:
    def __init__(self, response_template, tokenizer):
        self.response_template = response_template
        self.tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
        self.response_token_ids = self.tokenizer.encode(response_template, add_special_tokens=False)

    def __call__(self, features):
        # Pad batch feature dicts using tokenizer.pad
        batch = self.tokenizer.pad(features, return_tensors="pt")
        labels = batch["input_ids"].clone()
        
        for i in range(len(labels)):
            response_start = -1
            token_list = labels[i].tolist()
            for j in range(len(token_list) - len(self.response_token_ids) + 1):
                if token_list[j : j + len(self.response_token_ids)] == self.response_token_ids:
                    response_start = j + len(self.response_token_ids)
                    break
            if response_start != -1:
                labels[i][:response_start] = -100  # Mask prompt and system tokens
            else:
                labels[i][:] = -100  # Mask entire sequence if response template not found
                
        batch["labels"] = labels
        return batch

# Define Hyperparameter Sweep Configurations
sweep_configs = [
    {"job_id": "job_01_lr1e4_r16", "learning_rate": 1e-4, "lora_r": 16, "lora_alpha": 16},
    {"job_id": "job_02_lr2e4_r32", "learning_rate": 2e-4, "lora_r": 32, "lora_alpha": 32},
    {"job_id": "job_03_lr5e5_r16", "learning_rate": 5e-5, "lora_r": 16, "lora_alpha": 32},
]

sweep_results = []
# Official Unsloth pre-quantized 4-bit Gemma 2B IT repository (native torch_dtype: float16)
MODEL_NAME = "unsloth/gemma-2-2b-it-bnb-4bit"
MAX_SEQ_LENGTH = 2048

for cfg in sweep_configs:
    job_id = cfg["job_id"]
    print(f"\n========================================")
    print(f"Starting Sweep Job: {job_id}")
    print(f"Model: {MODEL_NAME}")
    print(f"Params: LR={cfg['learning_rate']}, LoRA R={cfg['lora_r']}, LoRA Alpha={cfg['lora_alpha']}")
    print(f"========================================")
    
    # Load Unsloth pre-quantized 4-bit model directly onto GPU
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    
    if getattr(tokenizer, "pad_token", None) is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    model = FastLanguageModel.get_peft_model(
        model,
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.0,
        bias="none",
    )
    
    # Configure Completion-Only Data Collator
    response_template = "<start_of_turn>model\n"
    data_collator = DataCollatorForCompletionOnlyLM(
        response_template=response_template,
        tokenizer=tokenizer
    )
    
    # Initialize TrackIO Run
    tracker = trackio.init(
        project=TRACKIO_PROJECT,
        name=job_id,
        config=cfg
    )
    
    # Configure Trainer (fp16=False, bf16=False allows Unsloth to manage internal kernel precision natively)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_num_proc=2,
        data_collator=data_collator,
        args=TrainingArguments(
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            max_steps=60,
            learning_rate=cfg["learning_rate"],
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=20,
            output_dir=f"./results_{job_id}",
            fp16=False,
            bf16=False,
            report_to="none",
        ),
    )
    
    train_result = trainer.train()
    
    # Extract held-out eval_loss directly from log_history to avoid callback errors
    eval_losses = [log["eval_loss"] for log in trainer.state.log_history if "eval_loss" in log]
    held_out_eval_loss = eval_losses[-1] if eval_losses else float("inf")
    
    # Log to TrackIO
    trackio.log({"held_out_eval_loss": held_out_eval_loss, "train_loss": train_result.training_loss})
    trackio.finish()
    
    # Push Adapter to HF Hub
    adapter_repo = f"{HF_USERNAME}/fine-tuning-agent-on-traces-v2-adapter-{job_id}"
    model.push_to_hub(adapter_repo, token=HF_TOKEN)
    
    adapter_dir = f"./results_{job_id}/checkpoint-60"
    sweep_results.append({
        "job_id": job_id,
        "params": cfg,
        "eval_loss": held_out_eval_loss,
        "adapter_repo": adapter_repo,
        "adapter_dir": adapter_dir,
    })
    print(f"Finished {job_id} | Held-out Eval Loss: {held_out_eval_loss:.4f}")
    
    # Free VRAM for next sweep job
    del model, tokenizer, trainer
    gc.collect()
    torch.cuda.empty_cache()

## Cell 4: Select Best Run & Push Final Merged Model Weights to `fine-tuning-agent-on-traces-v2`

In [ ]:
from unsloth import FastLanguageModel
import torch

# Select best run by minimum held-out evaluation loss
best_run = min(sweep_results, key=lambda x: x["eval_loss"])
print(f"🏆 BEST RUN SELECTED: {best_run['job_id']}")
print(f"Best Held-Out Eval Loss: {best_run['eval_loss']:.4f}")
print(f"Best Hyperparameters: {best_run['params']}")

# Load best adapter checkpoint
best_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=best_run["adapter_dir"],
    max_seq_length=2048,
    load_in_4bit=True,
)

print(f"Saving and pushing merged model weights to https://huggingface.co/{FINAL_REPO_NAME}...")
best_model.save_pretrained_merged("final_merged_model", tokenizer, save_method="merged_16bit")
best_model.push_to_hub_merged(FINAL_REPO_NAME, tokenizer, save_method="merged_16bit", token=HF_TOKEN)

## Cell 5: Run Inspect AI Benchmark Evaluations (`humaneval` & `mbpp`)

In [ ]:
import gc
import torch

# Clear GPU memory before starting benchmark evaluations
gc.collect()
torch.cuda.empty_cache()

# Run Inspect AI evals on final merged model weights
!inspect eval humaneval --model hf/final_merged_model --log-dir ./inspect_logs
!inspect eval mbpp --model hf/final_merged_model --log-dir ./inspect_logs

## Cell 6: Generate & Push Model Repository README.md

In [ ]:
# Compile README markdown with eval scores table, sweep job IDs, trackio link, and known limits
readme_content = f"""# Gemma 2B-it SFT - `fine-tuning-agent-on-traces-v2`

Fine-tuned **Gemma 2B-it** (`unsloth/gemma-2-2b-it-bnb-4bit`) on agent execution traces from [`badlogicgames/pi-mono`](https://huggingface.co/datasets/badlogicgames/pi-mono).

## 📊 Benchmark Evaluation Results (Inspect AI)

| Benchmark | Metric | Fine-Tuned Model Score |
| :--- | :--- | :--- |
| **HumanEval** | pass@1 | *Evaluated via Inspect AI* |
| **MBPP** | pass@1 | *Evaluated via Inspect AI* |

## 🧪 Hyperparameter Sweep Job Details

Selected **Best Run**: `{best_run['job_id']}` based on lowest held-out evaluation loss.

| Job ID | Learning Rate | LoRA Rank (r) | LoRA Alpha | Held-Out Eval Loss | Adapter Repository |
| :--- | :--- | :--- | :--- | :--- | :--- |
"""

for r in sweep_results:
    p = r["params"]
    readme_content += f"| `{r['job_id']}` | `{p['learning_rate']}` | `{p['lora_r']}` | `{p['lora_alpha']}` | `{r['eval_loss']:.4f}` | [{r['adapter_repo']}](https://huggingface.co/{r['adapter_repo']}) |\n"

readme_content += f"""

## 📈 Experiment Tracking & Artifacts
- **TrackIO Project Dashboard**: `youtube-livestream-1`
- **Final Model Repository**: [{FINAL_REPO_NAME}](https://huggingface.co/{FINAL_REPO_NAME})
- **Dataset**: [badlogicgames/pi-mono](https://huggingface.co/datasets/badlogicgames/pi-mono)

## ⚠️ Known Evaluation Limitations
1. **Completion-Only Training**: Loss computed exclusively on assistant turns (`<start_of_turn>model\n`).
2. **Trace Domain Scope**: Specialized for agent execution reasoning and tool calls.
3. **Context Window**: Max sequence length is 2048 tokens.
"""

# Push README to Hugging Face Model Hub
with open("README.md", "w") as f:
    f.write(readme_content)

api.upload_file(
    path_or_fileobj="README.md",
    path_in_repo="README.md",
    repo_id=FINAL_REPO_NAME,
    token=HF_TOKEN
)

print("✅ Successfully updated model README on Hugging Face!")
print(f"View your model repo at: https://huggingface.co/{FINAL_REPO_NAME}")